# PHASE 5 — Explicabilité SHAP
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Interpréter le modèle XGBoost final à l'aide de SHAP — explicabilité globale et locale.

**Jours 25-28 du plan**

---
### Contenu :
- Jours 25-26 : Explicabilité globale (summary_plot, bar_plot, top 20 variables)
- Jours 27-28 : Explications locales (force_plot pour 3 fraudes + 2 normales)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

print(f'SHAP version: {shap.__version__}')

In [ ]:
# === CHARGEMENT ===
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/FRAUDX/data/'
model = joblib.load('models/xgb_model.pkl')

X_test = pd.read_pickle(f'{data_path}X_test.pkl')
y_test = pd.read_pickle(f'{data_path}y_test.pkl')
best_threshold = np.load('models/best_threshold.npy')

if isinstance(y_test, pd.DataFrame):
    y_test = y_test.values.ravel()

print(f'Modèle chargé : {type(model).__name__}')
print(f'X_test : {X_test.shape}')

---
## JOURS 25-26 — Explicabilité globale

**Objectif :** Identifier les variables les plus importantes pour le modèle, produire summary_plot et bar_plot.

In [ ]:
# === Création de l'explainer SHAP ===
# Pour XGBoost, TreeExplainer est le plus adapté
explainer = shap.TreeExplainer(model)
print('✅ TreeExplainer créé')

In [ ]:
# === Calcul des valeurs SHAP (échantillon pour la vitesse) ===
# Utiliser un échantillon représentatif de X_test
X_sample = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(X_sample)
print(f'Valeurs SHAP calculées : {shap_values.shape}')

In [ ]:
# === Figure — SHAP Summary Plot (top 20 variables) ===
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('Figure 3.x — SHAP Summary Plot (Top 20 variables)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Figure — SHAP Bar Plot (importance moyenne) ===
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
plt.title('Figure 3.x — SHAP Feature Importance (moyenne des valeurs absolues)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Top 10 variables les plus importantes ===
shap_importance = np.abs(shap_values).mean(axis=0)
feature_names = X_sample.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': shap_importance
}).sort_values('importance', ascending=False)

print('=== TOP 10 VARIABLES (SHAP importance) ===')
for i, row in importance_df.head(10).iterrows():
    print(f'  {i+1}. {row["feature"]:30s} : {row["importance"]:.5f}')

**Interprétation pour le mémoire (section 3.3) :**

Les variables les plus importantes selon SHAP révèlent les facteurs clés qui influencent les décisions du modèle :
- Si `TransactionAmt` est dans le top : le montant est un indicateur fort de fraude
- Si `time_since_last_tx_card1` est dans le top : le délai depuis la dernière transaction est pertinent (indicateur SIM swap)
- Si `V*` ou `C*` ou `D*` dominent : les features engineering d'IEEE-CIS capturent des patterns comportementaux

Ces résultats répondent directement à **HS3** (l'explicabilité facilite l'adoption par les analystes).

---
## JOURS 27-28 — Explications locales

**Objectif :** Analyser individuellement 3 transactions frauduleuses et 2 transactions normales.

In [ ]:
# === Prédictions sur l'échantillon ===
y_probs_sample = model.predict_proba(X_sample)[:, 1]
y_pred_sample = (y_probs_sample >= best_threshold).astype(int)

# Vérité terrain correspondante
y_sample_true = y_test[X_sample.index]

print(f'Distribution dans l\'échantillon :')
print(f'  Vraies fraudes : {(y_sample_true == 1).sum()}')
print(f'  Prédites fraudes : {(y_pred_sample == 1).sum()}')

In [ ]:
# === Cas 1 : Transaction frauduleuse correctement détectée ===
fraud_detected = X_sample[(y_sample_true == 1) & (y_pred_sample == 1)]
if len(fraud_detected) >= 3:
    indices = fraud_detected.index[:3]
else:
    indices = fraud_detected.index[:len(fraud_detected)]

print(f'Analyse de {len(indices)} transactions frauduleuses détectées')
for idx in indices:
    print(f'\n--- Transaction {idx} (probabilité={y_probs_sample[X_sample.index == idx][0]:.4f}) ---')

In [ ]:
# === Force Plot pour 3 fraudes ===
for i, idx in enumerate(indices[:3]):
    idx_pos = list(X_sample.index).index(idx)
    plt.figure()
    shap.force_plot(explainer.expected_value, shap_values[idx_pos,:],
                    X_sample.iloc[idx_pos,:], matplotlib=True, show=False)
    plt.title(f'Figure 3.x — Explication SHAP locale — Transaction frauduleuse #{i+1}',
              fontweight='bold', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Cas 4-5 : Transactions normales (non frauduleuses) ===
normal_correct = X_sample[(y_sample_true == 0) & (y_pred_sample == 0)]
normal_indices = normal_correct.index[:2]

print(f'Analyse de {len(normal_indices)} transactions normales')
for i, idx in enumerate(normal_indices[:2]):
    idx_pos = list(X_sample.index).index(idx)
    plt.figure()
    shap.force_plot(explainer.expected_value, shap_values[idx_pos,:],
                    X_sample.iloc[idx_pos,:], matplotlib=True, show=False)
    plt.title(f'Figure 3.x — Explication SHAP locale — Transaction normale #{i+1}',
              fontweight='bold', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Waterfall Plot (alternative plus lisible) ===
# Pour la première fraude détectée
if len(indices) > 0:
    idx = indices[0]
    idx_pos = list(X_sample.index).index(idx)
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[idx_pos],
        base_values=explainer.expected_value,
        data=X_sample.iloc[idx_pos].values,
        feature_names=X_sample.columns
    ), max_display=15, show=False)
    plt.title('Figure 3.x — Waterfall SHAP — Transaction frauduleuse (détail)',
              fontweight='bold')
    plt.tight_layout()
    plt.show()

**Commentaires pour le mémoire (exemples de rédaction) :**

> *"Cette alerte a été déclenchée principalement par le montant élevé de la transaction (TransactionAmt) et le délai très court depuis la dernière transaction (time_since_last_tx_card1), suggérant un possible SIM swap."*

> *"Le graphique SHAP ci-dessus montre que la variable Vxxx contribue positivement au score de fraude, tandis que la variable Cyyy milite en faveur d'une transaction normale. Le modèle a correctement identifié cette transaction comme frauduleuse avec une probabilité de 0,97."*

> *"Ces visualisations SHAP sont conçues pour être lisibles par des non-spécialistes du Machine Learning, répondant ainsi aux exigences de la BCEAO/UEMOA en matière de transparence des décisions automatisées (HS3)."*

In [ ]:
# === Sauvegarde des figures ===
print('Les figures SHAP peuvent être sauvegardées avec plt.savefig().')
print('Exemple : plt.savefig("docs/shap_summary_plot.png", dpi=300, bbox_inches="tight")')

---
## Synthèse Phase 5 — Éléments pour le mémoire

### Figures produites :
1. **Figure 3.x** — SHAP Summary Plot (top 20 variables) → explicabilité globale
2. **Figure 3.x** — SHAP Bar Plot (importance moyenne) → variables les plus influentes
3. **Figures 3.x** — 3 SHAP Force Plots (transactions frauduleuses) → explicabilité locale
4. **Figures 3.x** — 2 SHAP Force Plots (transactions normales) → explicabilité locale
5. **Figure 3.x** — SHAP Waterfall Plot (détail d'une fraude) → alternative lisible

### Réponse à HS3 :
L'interprétabilité des modèles via SHAP facilite leur adoption par les analystes financiers
et les gestionnaires de risques en rendant les décisions du modèle transparentes et compréhensibles.
Ces visualisations peuvent également servir de support lors des entretiens qualitatifs (Phase 6).